# Error gallery

The pictures behind `09_error_analysis.ipynb`: every missed mass and every false
region, with the probabilities the model gave around it, so they can be looked at
and discussed one by one. Notebook 09 found that the model is all or nothing:
most missed masses get no response at all (a peak below 0.05), and found masses
and false regions alike peak near 1. This notebook shows what those masses and
regions look like.

Each row is one mass or one false region:

- **the whole breast**, with the annotated masses outlined in blue, the prediction
  at the chosen setting dashed in dark orange, and the zoomed area as a white box
- **the zoom without marks**, its contrast stretched to the zoomed area, to see
  the tissue as it is
- **the zoom with the probabilities**: orange, more opaque the higher the
  probability (nothing below `FLOOR`), the same outlines, and white crosses on the
  annotated findings that are not masses (calcifications, clusters, distortions
  and so on)
- for a missed mass, **the other view of the same breast**, when it is in the
  validation images: was the mass found there?

It needs the details file, the local `data/processed/inbreast/` and the checkpoints the
details were written with, `<RUNS>/fold_<k>/last.pt`. The images are predicted
again on the CPU (a few seconds each, a few minutes in all), and the predictions
are checked against the details file before anything is drawn. Download the
checkpoints of the whole-image run (about 300 MB each) with:

```
for k in 0 1 2 3 4; do
  kaggle kernels output giannispapatrechas/whole-vs-patch -p <folder> \
    --file-pattern "whole/fold_$k/last\.pt$"
done
```

and move `<folder>/runs/whole/fold_<k>/last.pt` to `RUNS`. The checkpoints and the
data are git-ignored.

**The figures show dataset images.** Outputs are stripped before commit
(`nbstripout`); do not save the figures in the repository or share them.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.path import Path as Polygon

from mammo_seg.data.dataset import MammogramDataset
from mammo_seg.data.manifest import read_manifest
from mammo_seg.data.paths import INBREAST_PROCESSED_DIR
from mammo_seg.data.preprocess_dataset import read_record
from mammo_seg.data.preprocessing import preprocess_points
from mammo_seg.data.schema import LesionType
from mammo_seg.data.strata import DEFAULT_SMALL_MASS_MM2
from mammo_seg.engine.checkpoint import load_checkpoint, model_from_checkpoint
from mammo_seg.engine.evaluate import (
    PostProcessing,
    image_infos,
    remove_small_regions,
    score_batch,
)
from mammo_seg.engine.inference import predict


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("pyproject.toml not found above " + str(start))


ROOT = find_repo_root(Path.cwd().resolve())
PROCESSED = ROOT / INBREAST_PROCESSED_DIR
RUNS = ROOT / "results" / "2026-10-24_whole_vs_patches" / "whole"
DETAILS = RUNS / "evaluation_validation_details.json"
THRESHOLD = 0.4  # the setting to show; it must be one of the details file
MIN_AREA_MM2 = 10.0
FLOOR = 0.05  # probabilities below this are not drawn
ZOOM_MM = 15.0  # the zoom reaches at least this far from the centre
NEAR_MM = 5.0  # a false region this close to an annotated finding is on it
CHEST_WALL = 0.1  # the strip next to the chest wall, as a fraction of the width
MAX_ROWS = None  # at most this many rows per group, or None for all
ROWS_PER_FIGURE = 5  # longer groups are split into several figures
BLUE, ORANGE, DARK_ORANGE = "#2a78d6", "#eb6834", "#a3401b"

## The details

As in notebook 09: the chosen setting of the details file, one row per mass and
per false region, and the peak response of each missed mass.

In [ ]:
details = json.loads(DETAILS.read_text())
manifest = read_manifest(PROCESSED / "manifest.json")
record = read_record(PROCESSED / "preprocessed" / "preprocessing.json")

setting = {"threshold": THRESHOLD, "min_area_mm2": MIN_AREA_MM2}
if setting not in details["settings"]:
    raise ValueError(f"{setting} is not in {DETAILS.name}: {details['settings']}")
SETTING = details["settings"].index(setting)
if "peak" not in json.dumps(details["images"][0]["results"][SETTING]):
    raise ValueError(f"{DETAILS.name} has no peaks: write it with the current code")
records = {image.file_id: image for image in manifest.images}
by_id = {image["file_id"]: image for image in details["images"]}

mass_rows, false_rows = [], []
for image in details["images"]:
    result = image["results"][SETTING]
    rows, cols = record.images[image["file_id"]].shape
    base = {
        "fold": image["fold"],
        "file_id": image["file_id"],
        "patient_id": image["patient_id"],
        "breast": records[image["file_id"]].laterality,
        "view": image["view"],
        "birads": image["birads"],
        "acr": image["acr"],
        "pixel_area_mm2": image["pixel_area_mm2"],
    }
    for index, mass in enumerate(result["masses"]):
        mass_rows.append(base | mass | {"index": index})
    for region in result["false_regions"]:
        false_rows.append(base | region | {"rel_col": region["col"] / cols})
masses = pd.DataFrame(mass_rows)
falses = pd.DataFrame(false_rows)
for frame in (masses, falses):
    frame["acr"] = frame.acr.astype("Int64")  # missing for a few images
masses["small"] = masses.area_mm2 < DEFAULT_SMALL_MASS_MM2
missed = masses[~masses.found].copy()
missed["response"] = pd.cut(
    missed.peak,
    [0, 0.05, 0.2, THRESHOLD, np.inf],
    right=False,
    labels=["none", "faint", "weak", "above the threshold"],
)
print(
    f"{len(missed)} missed masses on {missed.file_id.nunique()} images, "
    f"{len(falses)} false regions on {falses.file_id.nunique()} images"
)

## Predicting the images again

Only the images that are drawn are predicted: those with a missed mass or a false
region, the other views of the breasts with a missed mass, and the small masses
that were found, for comparison. Each image is predicted by the checkpoint of its
fold, as the evaluation did. The scores are then compared with the details file:
the Dice of every image and whether each mass was found. Small differences come
from the GPU and mixed precision of the evaluation; a large one means the
checkpoints are not the ones the details were written with, and stops here.

In [ ]:
def other_view(file_id: str) -> str | None:
    """The other view (CC or MLO) of the same breast among the details images."""
    image = by_id[file_id]
    for other in details["images"]:
        if (
            other["patient_id"] == image["patient_id"]
            and records[other["file_id"]].laterality == records[file_id].laterality
            and other["view"] != image["view"]
        ):
            return other["file_id"]
    return None


found_small = masses[masses.found & masses.small]
needed = set(missed.file_id) | set(falses.file_id) | set(found_small.file_id)
needed |= {v for v in map(other_view, missed.file_id) if v is not None}

post = PostProcessing(THRESHOLD, MIN_AREA_MM2)
infos = image_infos(record, manifest, sorted(needed))
# Only the image area is kept, in small types: the full samples of every image
# would not fit in the memory of a small machine.
pictures, truths, probabilities, cleaned = {}, {}, {}, {}
dice_gap, flipped = 0.0, []
torch.set_grad_enabled(False)
for fold in sorted({by_id[f]["fold"] for f in needed}):
    path = RUNS / f"fold_{fold}" / details["checkpoint"]
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing: download it (see the top)")
    checkpoint = load_checkpoint(path)
    if checkpoint.metadata.get("fold") != fold:
        raise ValueError(f"{path} is not the checkpoint of fold {fold}")
    if details["inference"][str(fold)] != "whole":
        raise ValueError(f"fold {fold} was not predicted in one pass")
    model = model_from_checkpoint(checkpoint).eval()
    ids = sorted(f for f in needed if by_id[f]["fold"] == fold)
    dataset = MammogramDataset(
        PROCESSED / "preprocessed", ids, checkpoint.preprocessing_settings
    )
    for index, file_id in enumerate(ids):
        sample = dataset[index]
        probability = predict(model, sample["image"][None])
        (result,) = score_batch(
            probability,
            sample["mask"][None],
            sample["valid"][None],
            post,
            [infos[file_id]],
        )
        reference = by_id[file_id]["results"][SETTING]
        dice_gap = max(dice_gap, abs(result.scores.dice - reference["dice"]))
        flipped += [
            file_id
            for mass, stored in zip(result.masses, reference["masses"], strict=True)
            if mass.found != stored["found"]
        ]
        rows, cols = record.images[file_id].shape
        keep = (probability[0, 0] >= THRESHOLD) & sample["valid"][0]
        keep = remove_small_regions(
            keep, post.min_pixels(infos[file_id].pixel_area_mm2)
        )
        pictures[file_id] = sample["image"][0, :rows, :cols].numpy().astype(np.float16)
        truths[file_id] = sample["mask"][0, :rows, :cols].numpy() > 0.5
        probabilities[file_id] = (
            probability[0, 0, :rows, :cols].numpy().astype(np.float16)
        )
        cleaned[file_id] = keep[:rows, :cols].numpy()
    del model, checkpoint
    print(f"fold {fold}: {len(ids)} images predicted")

print(f"largest Dice difference with the details: {dice_gap:.4f}")
print(f"masses found here but not there, or the reverse: {flipped or 'none'}")
if dice_gap > 0.05:
    raise ValueError("the predictions do not match the details: other checkpoints?")

## Drawing

`row` draws one mass or false region. The zoom is centred on it and reaches at
least `ZOOM_MM` from the centre, or one and a half times the size of a large mass.

In [ ]:
PROBABILITY = LinearSegmentedColormap.from_list(
    "probability", [(*plt.matplotlib.colors.to_rgb(ORANGE), 0.0), ORANGE]
)


def findings(file_id: str) -> np.ndarray:
    """The centres (x, y) of the annotated findings that are not masses."""
    params = record.images[file_id]
    centres = [
        preprocess_points(roi.points, params).mean(axis=0)
        for roi in records[file_id].rois
        if roi.lesion_type is not LesionType.MASS
    ]
    return np.array(centres).reshape(-1, 2)


def outlines(ax, file_id: str) -> None:
    if truths[file_id].any():
        ax.contour(truths[file_id], levels=[0.5], colors=BLUE, linewidths=1.2)
    if cleaned[file_id].any():
        ax.contour(
            cleaned[file_id],
            levels=[0.5],
            colors=DARK_ORANGE,
            linewidths=1.2,
            linestyles="--",
        )


def zoom_box(region: pd.Series) -> tuple[int, int, int, int]:
    """(top, bottom, left, right) of a square around ``region``, in the image."""
    mm = np.sqrt(region.pixel_area_mm2)
    radius = np.sqrt(region.area_mm2 / np.pi) / mm
    half = int(max(ZOOM_MM / mm, 1.5 * radius))
    rows, cols = record.images[region.file_id].shape
    top = int(np.clip(region.row - half, 0, max(rows - 2 * half, 0)))
    left = int(np.clip(region.col - half, 0, max(cols - 2 * half, 0)))
    return top, min(top + 2 * half, rows), left, min(left + 2 * half, cols)


def whole(ax, file_id: str, title: str, box=None) -> None:
    ax.imshow(pictures[file_id], cmap="gray", vmin=0, vmax=1)
    outlines(ax, file_id)
    if box is not None:
        top, bottom, left, right = box
        ax.add_patch(
            Rectangle(
                (left, top),
                right - left,
                bottom - top,
                fill=False,
                edgecolor="white",
                linewidth=1,
            )
        )
    ax.set_title(title, fontsize=8)
    ax.axis("off")


def zoom(ax, file_id: str, box, marks: bool) -> None:
    top, bottom, left, right = box
    crop = pictures[file_id][top:bottom, left:right].astype(np.float32)
    low, high = np.percentile(crop, [1, 99])
    extent = (left - 0.5, right - 0.5, bottom - 0.5, top - 0.5)
    ax.imshow(crop, cmap="gray", vmin=low, vmax=max(high, low + 1e-3), extent=extent)
    if marks:
        probability = probabilities[file_id][top:bottom, left:right]
        ax.imshow(
            np.ma.masked_less(probability, FLOOR),
            cmap=PROBABILITY,
            vmin=0,
            vmax=1,
            extent=extent,
        )
        outlines(ax, file_id)
        points = findings(file_id) - 0.5  # continuous to pixel coordinates
        inside = (
            (points[:, 0] >= left)
            & (points[:, 0] < right)
            & (points[:, 1] >= top)
            & (points[:, 1] < bottom)
        )
        ax.scatter(*points[inside].T, marker="+", s=40, color="white", linewidths=1)
        ax.set_xlim(left - 0.5, right - 0.5)
        ax.set_ylim(bottom - 0.5, top - 0.5)
    ax.axis("off")


def describe(region: pd.Series) -> str:
    acr = "?" if pd.isna(region.acr) else int(region.acr)
    return (
        f"{region.file_id} {region.breast} {region.view}, ACR {acr}, "
        f"BI-RADS {region.birads}\n{region.area_mm2:.0f} mm², "
        f"peak {region.peak:.2f}"
    )


def other_title(file_id: str | None) -> str:
    if file_id is None:
        return "no other view in the validation images"
    result = by_id[file_id]["results"][SETTING]
    found = sum(m["found"] for m in result["masses"])
    return (
        f"other view {by_id[file_id]['view']} ({file_id}): "
        f"{found} of {len(result['masses'])} masses found"
    )


LEGEND = [
    Line2D([], [], color=BLUE, label="annotated mass"),
    Line2D([], [], color=DARK_ORANGE, linestyle="--", label="prediction"),
    Line2D([], [], color=ORANGE, linewidth=6, label=f"probability ≥ {FLOOR}"),
    Line2D([], [], color="grey", marker="+", linestyle="", label="other finding"),
]


def gallery(frame: pd.DataFrame, title: str, with_other: bool = False) -> None:
    """One row per region of ``frame``, ``ROWS_PER_FIGURE`` rows per figure;
    nothing for an empty frame."""
    frame = frame if MAX_ROWS is None else frame.head(MAX_ROWS)
    if frame.empty:
        print(f"{title}: none")
        return
    columns = 4 if with_other else 3
    for first in range(0, len(frame), ROWS_PER_FIGURE):
        part = frame.iloc[first : first + ROWS_PER_FIGURE]
        fig, axes = plt.subplots(
            len(part), columns, figsize=(3.5 * columns, 3.7 * len(part)), squeeze=False
        )
        for (_, region), ax in zip(part.iterrows(), axes, strict=True):
            box = zoom_box(region)
            whole(ax[0], region.file_id, describe(region), box)
            zoom(ax[1], region.file_id, box, marks=False)
            zoom(ax[2], region.file_id, box, marks=True)
            if with_other:
                other = other_view(region.file_id)
                if other is None:
                    ax[3].axis("off")
                    ax[3].set_title(other_title(None), fontsize=8)
                else:
                    whole(ax[3], other, other_title(other))
        plt.tight_layout()
        fig.legend(
            handles=LEGEND, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=4
        )
        shown = f"{first + 1}-{first + len(part)} of {len(frame)}"
        fig.suptitle(f"{title} ({shown})", y=1.0 + 0.6 / (3.7 * len(part)))
        plt.show()
        plt.close(fig)

## 1. Missed masses

By response, as in notebook 09, large masses first. For a mass with no response,
the question is what makes it invisible to the model: is it small, hidden in
dense tissue, does it look like the tissue around it, or is it clearly visible
and simply not learned? The other view shows whether the same mass is found
there.

In [ ]:
ordered = missed.sort_values(
    ["response", "small", "area_mm2"], ascending=[True, True, False]
)
for response, group in ordered.groupby("response", observed=True):
    gallery(group, f"Missed masses, {response} response", with_other=True)

## 2. False regions

Grouped as in notebook 09: next to the chest wall (in the strip of `CHEST_WALL` of
the width), on or near an annotated finding that is not a mass (within
`NEAR_MM`), and the others. Look for what the model takes for a mass: the
pectoral muscle, lymph nodes, calcifications, dense tissue, the image border.

In [ ]:
def near_finding(region: pd.Series) -> bool:
    params = record.images[region.file_id]
    centre = (region.col + 0.5, region.row + 0.5)  # (x, y), continuous
    mm = np.sqrt(region.pixel_area_mm2)
    for roi in records[region.file_id].rois:
        if roi.lesion_type is LesionType.MASS:
            continue
        points = preprocess_points(roi.points, params)
        if len(points) >= 3 and Polygon(points).contains_point(centre):
            return True
        if float(np.hypot(*(points - centre).T).min()) * mm <= NEAR_MM:
            return True
    return False


falses["place"] = np.select(
    [falses.rel_col < CHEST_WALL, falses.apply(near_finding, axis=1)],
    ["at the chest wall", "on another finding"],
    "elsewhere",
)
for place in ("at the chest wall", "on another finding", "elsewhere"):
    group = falses[falses.place == place].sort_values("area_mm2", ascending=False)
    gallery(group, f"False regions {place}")

## 3. Small masses that were found

For comparison with the small masses that were missed: what does a small mass the
model does see look like?

In [ ]:
gallery(found_small.sort_values("area_mm2"), "Small masses found")

## Questions for the discussion

- Are the missed masses with no response visible to a reader at this resolution,
  in the zoom without marks? If the small ones are, the input resolution is
  probably not what hides them.
- Do the large missed masses look like the dense tissue around them, or like the
  found ones?
- Is a mass missed in one view clearly visible in the other, where it was found?
- Do the false regions at the chest wall lie on the pectoral muscle, or on lymph
  nodes in the axilla?
- Are the false regions near other findings on calcification clusters, or on
  distortions that look like a mass?